# References

## References for Address

In [ ]:
# Ghidra Scripting: References
# @category: GhidraScripting
# @author: Junjie Zhang

# enumerate all references from and to an address
addr = askAddress("Ghidra Scripting - References", "Please input an address:")
for i in getReferencesFrom(addr):
    print("a ref from this address: {}".format(i))
for i in getReferencesTo(addr):
    print("a ref to this address: {}".format(i))

In [ ]:
# Ghidra Scripting: References
# @category: GhidraScripting
# @author: Junjie Zhang

# enumerate all references from and to an address
addr = askAddress("Ghidra Scripting - References", "Please input an address:")

refManager = currentProgram.getReferenceManager()

for i in refManager.getReferencesFrom(addr):
    print("a ref from this address: {}".format(i))
for i in refManager.getReferencesTo(addr):
    print("a ref to this address: {}".format(i))

## References within Function

In [ ]:
# Ghidra Scripting: References
# @category: GhidraScripting
# @author: Junjie Zhang

myFunc = getFunctionContaining(currentAddress)
if myFunc:
    print(myFunc.getName())
    fbody = myFunc.getBody()
    for addr in fbody.getAddresses(True):
        for i in getReferencesFrom(addr):
            print("a ref from this address {}: {}".format(addr, i))
        for i in getReferencesTo(addr):
            print("a ref to this address {}: {}".format(addr, i))

## References for Instruction

In [ ]:
# Ghidra Scripting: References
# @category: GhidraScripting
# @author: Junjie Zhang

myFunc = getFunctionContaining(currentAddress)
if myFunc:
    inst = getFirstInstruction(myFunc)
    if inst:
        addr = inst.getAddress()
        for i in getReferencesFrom(addr):
            print("a ref from this address {}: {}".format(addr, i))
        for i in getReferencesTo(addr):
            print("a ref to this address {}: {}".format(addr, i))

## Reference Types

In [ ]:
# Ghidra Scripting: References
# @category: GhidraScripting
# @author: Junjie Zhang

myListing = currentProgram.getListing()
instructionIterator = myListing.getInstructions(True)
for inst in instructionIterator:
    addr = inst.getAddress()
    allRefsFromAddr = getReferencesFrom(addr)
    for ref in allRefsFromAddr:
        if ref.getReferenceType().isJump():
            print("{} with the specific type of {}".format(ref, ref.getReferenceType().getName()))

In [ ]:
# Ghidra Scripting: References
# @category: GhidraScripting
# @author: Junjie Zhang

from ghidra.program.model.symbol import RefType

myListing = currentProgram.getListing()
instructionIterator = myListing.getInstructions(True)
for inst in instructionIterator:
    addr = inst.getAddress()
    allRefsFromAddr = getReferencesFrom(addr)
    for ref in allRefsFromAddr:
        if ref.getReferenceType() == RefType.UNCONDITIONAL_JUMP:
            print("{} with the specific type of {}".format(ref, ref.getReferenceType().getName()))

## Callees using References

In [ ]:
# Ghidra Scripting: References
# @category: GhidraScripting
# @author: Junjie Zhang

# find all callees of the current function.
myFunc = getFunctionContaining(currentAddress)
if myFunc:
    print(myFunc)

    fbody = myFunc.getBody() # fbody is an object of AddressSetView
    myListing = currentProgram.getListing()
    instructionIterator = myListing.getInstructions(fbody, True)

    for inst in instructionIterator:
        addr = inst.getAddress()
        for ref in getReferencesFrom(addr):
            if ref.getReferenceType().isCall():
                calleeAddr = ref.getToAddress()
                calleeFunc = getFunctionAt(calleeAddr)
                print("{} at {} calls {}".format(myFunc, addr, calleeFunc))

## Callers using References

In [ ]:
# old implementation
callers = set()

myFunc = getFunctionContaining(currentAddress)
if myFunc:
    # getInstructions returns an iterator of instructions inside this binary
    myListing = currentProgram.getListing()
    instructionIterator = myListing.getInstructions(True)
    for inst in instructionIterator:
        if inst.getMnemonicString().startswith("CALL"):
            for calleeAddr in inst.getFlows():
                if myFunc.getEntryPoint() == calleeAddr:
                    callerFunc = getFunctionContaining(inst.getAddress())
                    print("Caller: {} at {} calls {}".format(callerFunc, inst.getAddress(), myFunc))
                    callers.add(callerFunc)

print(callers)

In [ ]:
# Ghidra Scripting: References
# @category: GhidraScripting
# @author: Junjie Zhang

# find all callers of the current function.

myFunc = getFunctionContaining(currentAddress)
if myFunc:
    entryPoint = myFunc.getEntryPoint()
    for ref in getReferencesTo(entryPoint):
        if ref.getReferenceType().isCall():
            callerInstAddr = ref.getFromAddress()
            callerFunc = getFunctionContaining(callerInstAddr)
            print("{} is called by {} at {}".format(myFunc, callerFunc, callerInstAddr))

## Functions with Loops

In [ ]:
# Ghidra Scripting: References
# @category: GhidraScripting
# @author: Junjie Zhang

# find all functions with loops.

funcsWithLoop = set()

myListing = currentProgram.getListing()
fm = currentProgram.getFunctionManager()
allFuncs = fm.getFunctions(True)

for f in allFuncs:
    f_body = f.getBody()

    instructionIterator = myListing.getInstructions(f_body, True)

    for inst in instructionIterator:
        allRefsFromInst = getReferencesFrom(inst.getAddress())
        allJumpRefsFromInst = filter(lambda x: x.getReferenceType().isJump(), allRefsFromInst)
        allBackwardJumpRefsFromInst = filter(lambda x: x.getFromAddress() > x.getToAddress(), allJumpRefsFromInst)
        # if len(allBackwardJumpRefsFromInst) > 0:
        # doesn't work because filter returns a filter object, not a list
        # so we need to convert it to a list first
        if len(list(allBackwardJumpRefsFromInst)) > 0:
            funcsWithLoop.add(f)

print("Functions with loop:")
for f in funcsWithLoop:
    print(f)

## Recursive Functions

In [ ]:
# Ghidra Scripting: References
# @category: GhidraScripting
# @author: Junjie Zhang

# find all functions with recursion.

funcsWithRecursion = set()

myListing = currentProgram.getListing()
fm = currentProgram.getFunctionManager()
allFuncs = fm.getFunctions(True)

for f in allFuncs:
    f_body = f.getBody()

    instructionIterator = myListing.getInstructions(f_body, True)

    for inst in instructionIterator:
        allRefsFromInst = getReferencesFrom(inst.getAddress())
        allCallRefsFromInst = filter(lambda x: x.getReferenceType().isCall(), allRefsFromInst)
        allSelfCallRefsFromInst = filter(lambda x: x.getToAddress() == f.getEntryPoint(), allCallRefsFromInst)
        # if len(allSelfCallRefsFromInst) > 0:
        # doesn't work because filter returns a filter object, not a list
        # so we need to convert it to a list first
        if len(list(allSelfCallRefsFromInst)) > 0:
            funcsWithRecursion.add(f)

print("Functions with recursion:")
for f in funcsWithRecursion:
    print(f)

In [ ]:
# Ghidra Scripting: References
# @category: GhidraScripting
# @author: Junjie Zhang

# find all functions with recursion.

funcsWithRecursion = set()

myListing = currentProgram.getListing()
fm = currentProgram.getFunctionManager()
allFuncs = fm.getFunctions(True)

for f in allFuncs:

    entryPoint = f.getEntryPoint()
    f_body = f.getBody()

    allRefsToEntryPoint = getReferencesTo(entryPoint)
    allCallRefsToEntryPoint = filter(lambda x: x.getReferenceType().isCall(), allRefsToEntryPoint)
    allSelfCallRefsToEntryPoint = filter(lambda x: f_body.contains(x.getFromAddress()), allCallRefsToEntryPoint)

    # if len(allSelfCallRefsToEntryPoint) > 0:
    # doesn't work because filter returns a filter object, not a list
    # so we need to convert it to a list first
    if len(list(allSelfCallRefsToEntryPoint)) > 0:
        funcsWithRecursion.add(f)

print("Functions with recursion:")
for f in funcsWithRecursion:
    print(f)